In [ ]:
# ========================
# PROJETO IA + REGRAS DE NEGÓCIO
# MATERIAL + NOME_CONCO -> CONVERSAO e FATOR
# ========================

import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# ---------- 1) Função para extrair número de texto ----------
def extrair_numero(texto):
    if pd.isna(texto):
        return 1
    numeros = re.findall(r'\d+', str(texto))
    return int(numeros[0]) if numeros else 1

# ---------- 2) Carregar base de treino ----------
tabela = pd.read_excel("Banco_de_Fardos.xlsx")
tabela.columns = tabela.columns.str.strip()
tabela = tabela[["MATERIAL", "NOME_CONCO", "CONVERSAO", "FATOR"]].copy()

# Garantir fator numérico
tabela["FATOR"] = pd.to_numeric(tabela["FATOR"], errors="coerce").fillna(0)

# Extrair números de MATERIAL e NOME_CONCO
tabela["NUM_MATERIAL"] = tabela["MATERIAL"].apply(extrair_numero)
tabela["NUM_NOMECONC"] = tabela["NOME_CONCO"].apply(extrair_numero)

# ---------- 3) Codificação para IA ----------
enc_m = LabelEncoder(); tabela["MATERIAL_enc"] = enc_m.fit_transform(tabela["MATERIAL"].astype(str))
enc_n = LabelEncoder(); tabela["NOME_enc"]     = enc_n.fit_transform(tabela["NOME_CONCO"].astype(str))
enc_conv = LabelEncoder(); tabela["CONV_enc"] = enc_conv.fit_transform(tabela["CONVERSAO"].astype(str))

# Features e targets
X = tabela[["MATERIAL_enc", "NOME_enc", "NUM_MATERIAL", "NUM_NOMECONC"]]
y_conv = tabela["CONV_enc"]
y_fator = tabela["FATOR"]

# ---------- 4) Treinar modelos ----------
modelo_conv = RandomForestClassifier(random_state=42, n_estimators=200)
modelo_conv.fit(X, y_conv)

modelo_fator = RandomForestRegressor(random_state=42, n_estimators=200)
modelo_fator.fit(X, y_fator)

print("✅ Modelos treinados")

# ---------- 5) Carregar novos fardos ----------
novos = pd.read_excel("novos_fardos.xlsx")
novos.columns = novos.columns.str.strip()
novos = novos[["MATERIAL", "NOME_CONCO"]].copy()

# Extrair números
novos["NUM_MATERIAL"] = novos["MATERIAL"].apply(extrair_numero)
novos["NUM_NOMECONC"] = novos["NOME_CONCO"].apply(extrair_numero)

# Mapear valores para encoders
def safe_map(series, encoder):
    m = {v:i for i,v in enumerate(encoder.classes_)}
    return series.astype(str).map(m).fillna(-1).astype(int)

novos["MATERIAL_enc"] = safe_map(novos["MATERIAL"], enc_m)
novos["NOME_enc"] = safe_map(novos["NOME_CONCO"], enc_n)

X_novos = novos[["MATERIAL_enc", "NOME_enc", "NUM_MATERIAL", "NUM_NOMECONC"]]

# ---------- 6) Previsões com IA ----------
pred_conv_enc = modelo_conv.predict(X_novos)
pred_conv = enc_conv.inverse_transform(
    np.clip(pred_conv_enc, 0, len(enc_conv.classes_)-1)
)

pred_fator = modelo_fator.predict(X_novos)

# ---------- 7) Regras de coerência ----------
resultado = novos.copy()
resultado["CONVERSAO"] = pred_conv
resultado["FATOR"] = pred_fator

# Regra determinística baseada em números
for i, row in resultado.iterrows():
    num_m = row["NUM_MATERIAL"]
    num_n = row["NUM_NOMECONC"]

    if num_m > 1 and (num_n == 1 or num_n == 0):
        resultado.at[i, "CONVERSAO"] = "multiplica"
        resultado.at[i, "FATOR"] = num_m
    elif num_n > 1 and (num_m == 1 or num_m == 0):
        resultado.at[i, "CONVERSAO"] = "divide"
        resultado.at[i, "FATOR"] = num_n
    elif num_m <= 1 and num_n <= 1:
        resultado.at[i, "CONVERSAO"] = "excluir"
        resultado.at[i, "FATOR"] = 0

# Forçar consistência final
resultado.loc[resultado["CONVERSAO"] == "excluir", "FATOR"] = 0
resultado.loc[~resultado["CONVERSAO"].isin(["multiplica", "divide", "excluir"]), ["CONVERSAO","FATOR"]] = ["desconhecido", 0]

# ---------- 8) Salvar resultado ----------
resultado.to_excel("PlanilhaAtualizada_Com_Regras.xlsx", index=False)
print("📂 Resultado salvo em PlanilhaAtualizada_Com_Regras.xlsx")
